# Transcriptomic classification audit

Execute from top to bottom to reclassify the discovery and validation cohorts from their gene-expression matrices. Each method has a separate code cell, exported inputs, scores, labels, source hashes and a before/after comparison. The final cell can replace the authoritative case tables after explicit opt-in and verification; it creates a timestamped AnnData backup.

LymphoMAP and EcoTyper call native author implementations. Kotlov and Ciavarella reproduce the study's explicitly specified clustering procedures; these are cohort-dependent reconstructions. Lymph2Cx is a **research reconstruction**, using the full 15 predictors plus five housekeeping genes, with externally frozen NanoString borders. It is not the proprietary clinical assay.

Portable defaults are declared in `config/classification_audit/default.json`; optional machine-specific overrides belong in the ignored `local.json` beside it. `promote` defaults to false. See `docs/classification-resources.md` for native software and model resources. Missing native software or source data is an error, never a reason to reuse saved labels.

In [1]:
from pathlib import Path
import sys, json, shutil, importlib.util, inspect
import numpy as np
import pandas as pd
import h5py
from anndata.io import read_elem
ROOT = Path.cwd() if (Path.cwd()/'dlbcl').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from dlbcl.classification_audit import (sha256, run, read_study, canonicalizer,
    preprocess_expression, promote_calls, CASE_KEYS)
from dlbcl.classification_audit import load_audit_config, require_audit_resources
config, CONFIG_PATH = load_audit_config(ROOT)
SOURCE = Path(config['source_adata']).resolve()
SOURCE_HASH = sha256(SOURCE)
tables, raw_expression = read_study(SOURCE)
canon = canonicalizer(config['hgnc'])
UPDATE_ADATA = config['promote']  # Explicit final-cell write switch; backup and atomic replacement.

require_audit_resources(config, 'transcriptomic')
OUT=Path(config['output_root'])/'transcriptomic'
OUT.mkdir(parents=True,exist_ok=True)
SEED=config['seed']; N_FEATURES=config['lymphomap_features']
POSTERIOR_CUTOFF=config['lymphomap_posterior_cutoff']
GCB_UNC_THRESHOLD=config['lymph2cx_lower']; UNC_ABC_THRESHOLD=config['lymph2cx_upper']
assert GCB_UNC_THRESHOLD < UNC_ABC_THRESHOLD
METHOD_FILES={str(p):sha256(p) for p in [CONFIG_PATH,ROOT/'dlbcl/classification_audit.py',ROOT/'dlbcl/rna_clustering_benchmark.py',ROOT/'dlbcl/lymph2cx_research.py',*sorted((ROOT/'config/classification_audit').glob('*.csv')),*sorted((ROOT/'config/classification_audit').glob('*.json'))]}
display(pd.DataFrame({'cohort':tables.keys(),'cases':[len(x) for x in tables.values()]}))


,cohort,cases
0,Discovery,64
1,Validation,303


## 1. Input scale and preprocessing

| Declared input | Processing before classification |
|---|---|
| Raw NanoString counts | Require probe classes and controls; positive-control scaling; explicitly chosen negative-control background correction; clipping only when background correction is selected; housekeeping normalization; explicit log2 offset. Save every sample's factors. |
| Normalized NanoString counts | No second background/HK normalization; explicit log2 transform for score/clustering methods. Native LymphoMapR also offers `NanoString_norm_counts`, which performs its own VST. |
| Normalized NanoString log2 | Retain supplied values. This is the study input; negative log ratios are valid. |
| TPM | `log2(TPM+1)`; never infer TPM from read counts. |
| log2(TPM+1) | Retain values. |
| Normalized microarray log2 | Retain values. Raw arrays require platform-specific preprocessing and probe-to-gene mapping upstream. |

The current study's validation pipeline uses positive-control scaling and log2 housekeeping centering **without negative-control subtraction**. Reproduce and verify that pipeline below. Discovery raw-control processing is not reconstructed here; its stored normalized matrix is the declared starting point. Adding background correction would be a separate preprocessing analysis, not a retrospective description of these matrices.

Native LymphoMapR's already-log branch is named `RNAseq_TPM_log2p1`; using it for normalized NanoString log ratios is the study's explicit cross-platform choice. Do not pass these negative values to `NanoString_norm_counts`. Microarrays use `microarray_lognorm`. Lymph2Cx NanoString thresholds must not be transferred automatically to RNA-seq or arrays.

In [2]:
# Inspect the exact supported transforms, including the raw-control implementation.
print(inspect.getsource(preprocess_expression))
# Verify current validation preprocessing from its embedded raw probes.
from dlbcl.validation_cohort import normalize_nanos_string_gep, gene_expression_matrix
with h5py.File(SOURCE,'r') as handle:
    raw_probes=read_elem(handle['uns/validation_cohort/gep_raw_probes'])
replayed=gene_expression_matrix(normalize_nanos_string_gep(raw_probes))
stored=raw_expression['Validation']
np.testing.assert_allclose(replayed.loc[stored.index,stored.columns], stored, atol=1e-10)
(OUT/'preprocessing_verification.json').write_text(json.dumps({
    'validation_raw_replay_max_abs_difference':float(np.max(np.abs(replayed.loc[stored.index,stored.columns].values-stored.values))),
    'background_subtraction':'not performed in the verified validation pipeline',
    'discovery_starting_point':'stored normalized log2 matrix; raw-control provenance not independently replayed'},indent=2))
expression={}; symbol_rows=[]; input_hashes={}
for cohort,x in raw_expression.items():
    x,normalization=preprocess_expression(x,config['study_input_kind'])
    normalization.to_csv(OUT/f'{cohort}_normalization.csv')
    symbol_rows.extend({'cohort':cohort,'input_gene':g,'canonical_gene':canon(g)} for g in x.index)
    x.index=[canon(g) for g in x.index]
    x=x.groupby(level=0,sort=False).mean() # Explicit alias-collision rule on log scale.
    x.columns=[f'{cohort}__{s}' for s in x.columns]
    expression[cohort]=x
    path=OUT/f'{cohort}_canonical_log.tsv';x.to_csv(path,sep='\t',float_format='%.17g')
    input_hashes[cohort]=sha256(path)
pd.DataFrame(symbol_rows).to_csv(OUT/'gene_symbol_mapping.csv',index=False)
updates={c:pd.DataFrame(index=[s.split('__',1)[1] for s in x.columns]) for c,x in expression.items()}


def preprocess_expression(x, kind, *, background=None, housekeeping=None, floor=1., pseudocount=None):
    """Return (log2 gene matrix, per-sample normalization audit).

    Raw NanoString requires Code.Class and Probe columns, explicit background
    choice, housekeeping genes, floor and pseudocount. This is a transparent
    research normalization, not a replacement for manufacturer's assay QC.
    Already normalized inputs NEVER receive background correction a second time.
    """
    if kind == 'nanostring_raw_counts':
        if background not in {'none','negative_mean','negative_mean_plus_2sd'} or not housekeeping:
            raise ValueError('Raw counts require explicit background and housekeeping choices')
        if pseudocount is None or pseudocount < 0 or floor <= 0:
            raise ValueError('Declare nonnegative pseudocount and positive background floor')
        if not {'Code.Class','Probe'} <= set(x): raise ValueError('Missing probe annotations')
        values=x.drop

## 2. LymphoMAP: native feature selection and naive Bayes

Run LymphoMapR afresh with the declared seed, feature budget and posterior cutoff. The package selects DEGs from genes shared with its reference, performs its native batch adjustment and fits/predicts its naive Bayes model. Export the selected features and full R result. [Li et al., Cancer Cell 2025](https://pmc.ncbi.nlm.nih.gov/articles/PMC12417684/).

In [3]:
lymphomap_r = r'''args <- commandArgs(TRUE)
suppressPackageStartupMessages(library(LymphoMapR))
set.seed(as.integer(args[4]))
x <- as.matrix(read.delim(args[1],row.names=1,check.names=FALSE))
r <- run_LymphoMap(x,data_type=args[3],n_features_per_archetype=as.integer(args[5]),
                  LymphoMAP_prob_cutoff=as.numeric(args[6]),verbose=FALSE)
write.table(r$pred,file.path(args[2],'predictions.tsv'),sep="\t",quote=FALSE,row.names=FALSE)
write.table(r$data$feature,file.path(args[2],'selected_features.tsv'),sep="\t",quote=FALSE,row.names=FALSE)
saveRDS(r,file.path(args[2],'result.rds'))
capture.output(sessionInfo(),file=file.path(args[2],'sessionInfo.txt'))
capture.output(LymphoMapR:::data_prep_expr,file=file.path(args[2],'native_preprocessing.txt'))
'''
script=OUT/'run_lymphomap.R';script.write_text(lymphomap_r)
reference=pd.read_csv(config['lymphomap_reference_genes'],sep='\t').gene.tolist()
for cohort,x in expression.items():
    work=OUT/'LymphoMAP'/cohort;work.mkdir(parents=True,exist_ok=True)
    genes=[g for g in reference if canon(g) in x.index]
    native=x.loc[[canon(g) for g in genes]].copy();native.index=genes
    native_input_type='microarray_lognorm' if config['study_input_kind']=='microarray_log2_normalized' else 'RNAseq_TPM_log2p1'
    path=work/'input.tsv';native.to_csv(path,sep='\t',index_label='Gene',float_format='%.12g')
    # Already-log input: retain values, with native ComBat and reference feature selection.
    run([config['rscript'],script,path,work,native_input_type,SEED,N_FEATURES,POSTERIOR_CUTOFF],work/'run.log')
    pred=pd.read_csv(work/'predictions.tsv',sep='\t').set_index('sample_id')
    assert set(pred.index)==set(x.columns)
    updates[cohort]['lymphomap']=pred.loc[x.columns,'LymphoMAP'].to_numpy()
    display(pred.LymphoMAP.value_counts().rename(cohort))


LymphoMAP
LN      28
FMAC    19
TEX     17
Name: Discovery, dtype: int64

LymphoMAP
LN      123
TEX      94
FMAC     86
Name: Validation, dtype: int64

## 3. EcoTyper: native state/ecotype recovery

The author recovery code scales reference genes within the query cohort and fills absent reference genes with zero. It is therefore cohort dependent. Native ecotype assignment can remain Unassigned; maximum abundance is stored separately and is not a probability or a confident assignment. Each run has an isolated workspace and exact source/model fingerprints. [EcoTyper code](https://github.com/digitalcytometry/ecotyper).

In [4]:
source=Path(config['ecotyper_source']); work=OUT/'ecotyper_work'
work.mkdir(exist_ok=True)
shutil.copytree(source/'pipeline',work/'pipeline',dirs_exist_ok=True)
shutil.copy2(source/'EcoTyper_recovery_bulk.R',work/'EcoTyper_recovery_bulk.R')
for part in ['Analysis','Cell_States/discovery','Ecotypes/discovery']:
    target=work/'EcoTyper/Lymphoma/Lymphoma_Fractions'/part
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists(): target.symlink_to((source/'EcoTyper/Lymphoma/Lymphoma_Fractions'/part).resolve(),target_is_directory=True)
model_hashes={str(p.relative_to(source)):sha256(p) for folder in ['pipeline','EcoTyper/Lymphoma/Lymphoma_Fractions/Cell_States/discovery','EcoTyper/Lymphoma/Lymphoma_Fractions/Ecotypes/discovery'] for p in (source/folder).rglob('*') if p.is_file()}
(OUT/'ecotyper_source_hashes.json').write_text(json.dumps(model_hashes,indent=2))
reference=sorted(pd.read_csv(config['ecotyper_reference_genes']).reference_gene.unique())
for cohort,x in expression.items():
    genes=[g for g in reference if canon(g) in x.index]
    native=x.loc[[canon(g) for g in genes]].copy();native.index=genes
    path=OUT/f'audit_{cohort}.tsv';native.to_csv(path,sep='\t',index_label='Gene',float_format='%.12g')
    destination=OUT/'EcoTyper'/cohort;destination.mkdir(parents=True,exist_ok=True)
    run([config['rscript'],work/'EcoTyper_recovery_bulk.R','-d','Lymphoma','-m',path,'-t','2','-o',destination],destination/'run.log',cwd=work)
    recovery=work/'EcoTyper/Lymphoma/Lymphoma_Fractions/Ecotypes/recovery'/path.stem
    assignment=pd.read_csv(recovery/'initial_ecotype_assignment.txt',sep='\t').set_index('ID')
    abundance=pd.read_csv(recovery/'ecotype_abundance.txt',sep='\t',index_col=0)
    normalized_ids=[s.replace('-','.') for s in x.columns]
    assert len(set(normalized_ids))==len(normalized_ids)
    assert set(abundance.columns)==set(normalized_ids)
    call=assignment.Ecotype.reindex(normalized_ids).fillna('Unassigned')
    updates[cohort]['Lymphoma_Ecotype']=call.to_numpy()
    updates[cohort]['Lymphoma_Ecotype_confident']=call.where(call.ne('Unassigned'),'').to_numpy()
    updates[cohort]['Lymphoma_Ecotype_imputed']=False
    updates[cohort]['New_Ecotype']=abundance[normalized_ids].idxmax().where(abundance[normalized_ids].max().gt(0),'').to_numpy()
    assignment.to_csv(destination/'assignment_with_q.csv');abundance.to_csv(destination/'abundance.csv')
    display(call.value_counts().rename(cohort))


Ecotype
LE6           18
Unassigned    12
LE4            6
LE8            6
LE3            5
LE9            5
LE1            4
LE7            4
LE2            2
LE5            2
Name: Discovery, dtype: int64

Ecotype
Unassigned    95
LE6           62
LE4           38
LE9           25
LE8           17
LE7           17
LE3           17
LE1           17
LE2            9
LE5            6
Name: Validation, dtype: int64

## 4. Kotlov and Ciavarella: explicit clustering and subtype annotation

Kotlov uses 22 author ssGSEA signatures plus NFkB/p53/PI3K PROGENy scores. Missing PROGENy genes contribute zero; signature coverage is exported. Median scaling uses the author's **mean absolute deviation about the mean**, then clips to ±3. The Louvain threshold grid and selection criteria are printed below. Cluster profiles are mapped to the highest-correlated external published subtype centroid, permitting many-to-one mapping. The stored centroids were built from panel-restricted Schmitz, Morin and Li references; provenance is supplied alongside them. This is not the author's KNN classifier.

Ciavarella uses the 45-gene Euclidean/Ward reconstruction, cut into three groups and ordered by average expression. These procedures depend on the query cohort; adding or removing patients can change assignments. [Kotlov et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC8178179/).

In [5]:
from dlbcl.rna_clustering_benchmark import author_median_scale, select_louvain, ward_partition
print(inspect.getsource(select_louvain))
spec=importlib.util.spec_from_file_location('kotlov_author_utils',config['kotlov_utils'])
author=importlib.util.module_from_spec(spec);spec.loader.exec_module(author)
features=pd.read_csv(ROOT/'config/classification_audit/clustering_features.csv')
reference=pd.read_csv(ROOT/'config/classification_audit/kotlov_reference_centroids.csv',index_col=0)
display_names={'LME-DE':'Depleted','LME-GC':'GC-like','LME-ME':'Mesenchymal','LME-IN':'Inflammatory'}
coverage=[]; mappings=[]
for cohort,x in expression.items():
    raw={}
    for name,group in features[features.classifier.eq('Kotlov')].groupby('signature',sort=False):
        g=group.copy();g['gene']=g.gene.map(canon)
        genes=list(dict.fromkeys(g.gene)); overlap=[v for v in genes if v in x.index]
        coverage.append(dict(cohort=cohort,signature=name,required=len(genes),measured=len(overlap)))
        if g.weight.isna().all():
            raw[name]=author.ssgsea_formula(x.T,{name:author.GeneSet(name,'author signature',genes)})[name]
        else:
            weights=g.groupby('gene',sort=False).weight.sum()
            raw[name]=x.T.reindex(columns=weights.index,fill_value=0).dot(weights)
    scores=pd.DataFrame(raw,index=x.columns)
    scaled=author_median_scale(scores)
    labels,threshold,selection=select_louvain(scaled,seed=SEED)
    selection.to_csv(OUT/f'{cohort}_kotlov_threshold_selection.csv',index=False)
    scores.to_csv(OUT/f'{cohort}_kotlov_scores.csv')
    clusters=pd.Series(['C'+str(v) for v in labels],index=x.columns)
    named={}
    for cluster,profile in scaled.groupby(clusters).mean().iterrows():
        correlations=reference.apply(lambda r:np.corrcoef(profile.reindex(reference.columns),r)[0,1],axis=1).sort_values(ascending=False)
        assert np.isfinite(correlations).all()
        named[cluster]=display_names[correlations.index[0]]
        mappings.append(dict(cohort=cohort,cluster=cluster,label=named[cluster],correlation=correlations.iloc[0],margin=correlations.iloc[0]-correlations.iloc[1]))
    updates[cohort]['KotlovSig']=clusters.map(named).to_numpy()
    updates[cohort]['Kotlov_cluster_id']=clusters.to_numpy()
    genes=list(dict.fromkeys(features[features.classifier.eq('Ciavarella')].gene.map(canon)))
    if not set(genes)<=set(x.index): raise ValueError('Ciavarella requires all declared 45 features')
    calls=ward_partition(x.loc[genes].T,standardize=False)
    updates[cohort]['Ciav_Cluster']=[{'High':'Hot','Intermediate':'Intermediate','Low':'Cold'}[v] for v in calls]
pd.DataFrame(coverage).to_csv(OUT/'kotlov_feature_coverage.csv',index=False)
pd.DataFrame(mappings).to_csv(OUT/'kotlov_subtype_mapping.csv',index=False)
display(pd.DataFrame(mappings))


def select_louvain(scores, seed=SEED, thresholds=THRESHOLDS):
    rows, partitions = [], {}
    for threshold in thresholds:
        labels = louvain_partition(scores, float(threshold), seed)
        counts = np.unique(labels, return_counts=True)[1]
        valid = 1 < len(counts) < len(scores) and counts.min() / len(scores) >= .05
        row = dict(threshold=float(threshold), clusters=len(counts), min_cluster_n=int(counts.min()), valid=valid)
        if valid:
            row.update(davies_bouldin=davies_bouldin_score(scores, labels),
                       calinski_harabasz=calinski_harabasz_score(scores, labels),
                       silhouette=silhouette_score(scores, labels, metric='euclidean'))
        rows.append(row)
        partitions[float(threshold)] = labels
    table = pd.DataFrame(rows)
    valid = table[table.valid].copy()
    if valid.empty:
        raise ValueError('No threshold yields eligible clusters; do not silently force a solution')
    valid['criterion_rank_s

,cohort,cluster,label,correlation,margin
0,Discovery,C1,Depleted,0.399206,0.390466
1,Discovery,C2,Depleted,0.509393,0.461147
2,Discovery,C3,Mesenchymal,0.709571,0.376938
3,Discovery,C4,Inflammatory,0.726147,0.557961
4,Validation,C1,Inflammatory,0.763225,0.788660
5,Validation,C2,Depleted,0.311952,0.173563
6,Validation,C3,Mesenchymal,0.747358,0.309104
7,Validation,C4,Depleted,0.735017,0.987893


## 5. Full Lymph2Cx research reconstruction

Apply the fixed 15-predictor coefficients after subtracting the mean log2 expression of five housekeeping genes. Every input gene is required. This study matrix contains ISY1 and RAB29, so no read-through-gene proxy or dropped predictor is used. The two displayed borders were calibrated externally on GSE178394 NanoString data; no study labels enter the fit. Gene-specific codeset calibration and proprietary QC remain unavailable. [Public calibration cohort](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE178394).

In [6]:
model=json.loads((ROOT/'config/classification_audit/lymph2cx_full.json').read_text())
f=pd.DataFrame(model['features']).set_index('gene')
assert len(f)==20 and int(f.normalization_weight.ne(0).sum())==5
score_rows=[]
for cohort,x in expression.items():
    if config['study_input_kind'] not in {'nanostring_log2_normalized','nanostring_normalized_counts','nanostring_raw_counts'}:
        raise ValueError('NanoString borders cannot automatically classify TPM/microarray inputs')
    if not set(f.index)<=set(x.index): raise ValueError('Missing full Lymph2Cx features; do not silently drop predictors')
    q=x.loc[f.index].T
    hk=q.mul(f.normalization_weight,axis=1).sum(axis=1)
    contribution=q.sub(hk,axis=0).mul(f.coefficient,axis=1)
    score=contribution.sum(axis=1)
    call=np.where(score<GCB_UNC_THRESHOLD,'GCB',np.where(score<UNC_ABC_THRESHOLD,'UNC','ABC'))
    updates[cohort]['COO_NanoString']=call
    contribution.to_csv(OUT/f'{cohort}_lymph2cx_gene_contributions.csv')
    score_rows.append(pd.DataFrame({'cohort':cohort,'query_id':q.index,'score':score.values,'call':call,'lower':GCB_UNC_THRESHOLD,'upper':UNC_ABC_THRESHOLD}))
scores=pd.concat(score_rows);scores.to_csv(OUT/'lymph2cx_scores.csv',index=False)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
for ax,(cohort,d) in zip(axes,scores.groupby('cohort',sort=False)):
    ax.hist(d.score,bins=25,color='#497999');ax.axvline(GCB_UNC_THRESHOLD,color='black',ls='--');ax.axvline(UNC_ABC_THRESHOLD,color='black',ls='--')
    ax.set(title=cohort,xlabel='Full Lymph2Cx research score',ylabel='Cases')
fig.tight_layout();fig.savefig(OUT/'lymph2cx_thresholds.pdf');plt.show()


/var/folders/k_/ffwscldj7gx7ch29c5ffhr4c0000gn/T/ipykernel_9245/3262331652.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'lymph2cx_thresholds.pdf');plt.show()


## 6. Review and replace the authoritative calls

Updates use current patient IDs directly from the canonical expression matrices. The transaction verifies unchanged clinical fields, genomic calls, spatial archetypes, expression and cell-level payloads. The native EcoTyper confidence and maximum-abundance fields are refreshed together. The source hash prevents promotion after another process changes the AnnData.

In [7]:
comparisons=[]
for cohort,new in updates.items():
    old=tables[cohort].copy()
    if 'patient_id' in old: old=old.set_index('patient_id',drop=False)
    assert set(new.index)==set(old.index) and not new.isna().any().any()
    new.to_csv(OUT/f'{cohort}_fresh_calls.csv')
    for col in new:
        prior=old[col].astype(str) if col in old else pd.Series('',index=old.index)
        comparisons.append(dict(cohort=cohort,column=col,n=len(new),changed=int(prior.reindex(new.index).ne(new[col].astype(str)).sum())))
display(pd.DataFrame(comparisons))
# Keep classifier-derived B-cell state summaries synchronized with native recovery.
auxiliary={}
for cohort in updates:
    prefix='uns' if cohort=='Discovery' else 'uns/validation_cohort'
    key=prefix+'/ecotyper_b_state'
    with h5py.File(SOURCE,'r') as h: old_state=read_elem(h[key])
    base=OUT/'ecotyper_work/EcoTyper/Lymphoma/Lymphoma_Fractions/Cell_States/recovery'/f'audit_{cohort}'/'B.cells/6'
    abundance=pd.read_csv(base/'state_abundances.txt',sep='\t',index_col=0)
    ids=[f'{cohort}__{pid}' for pid in updates[cohort].index]
    a=abundance[ids]; ordered=np.sort(a.to_numpy(),axis=0)
    fresh=pd.DataFrame({'Dominant_B_cell_state':a.idxmax().values,'Dominant_score':ordered[-1],
        'Second_score':ordered[-2],'Margin':ordered[-1]-ordered[-2]},index=updates[cohort].index)
    fresh.index.name=old_state.index.name
    # Keep ancillary/control samples outside the canonical cohort unchanged.
    replacement=pd.concat([old_state.loc[~old_state.index.isin(fresh.index)],fresh])
    replacement=replacement.reindex(old_state.index)
    for col in ['Dominant_score','Second_score','Margin']:
        replacement[col]=pd.to_numeric(replacement[col],errors='raise')
    auxiliary[key]=replacement

provenance={'config':config,'inputs':input_hashes,'method_files':METHOD_FILES,
            'native_LymphoMAP_code':sha256(OUT/'run_lymphomap.R'),
            'ecotyper_manifest':sha256(OUT/'ecotyper_source_hashes.json'),
            'method_status':{'Kotlov':'study clustering reconstruction + external centroid naming','Ciavarella':'study Ward reconstruction','Lymph2Cx':'full research reconstruction; frozen external NanoString borders'}}
(OUT/'run_provenance.json').write_text(json.dumps(provenance,indent=2))
if UPDATE_ADATA:
    changes=promote_calls(SOURCE,updates,OUT/'promotion',audit_name='transcriptomic',run_provenance=provenance,expected_source_sha256=SOURCE_HASH,auxiliary_updates=auxiliary)
    display(changes.groupby(['cohort','column']).changed.agg(['size','sum']))
else:
    print('Review-only run; UPDATE_ADATA is False.')


,cohort,column,n,changed
0,Discovery,lymphomap,64,0
1,Discovery,Lymphoma_Ecotype,64,0
2,Discovery,Lymphoma_Ecotype_confident,64,0
3,Discovery,Lymphoma_Ecotype_imputed,64,0
4,Discovery,New_Ecotype,64,0
5,Discovery,KotlovSig,64,0
6,Discovery,Kotlov_cluster_id,64,0
7,Discovery,Ciav_Cluster,64,0
8,Discovery,COO_NanoString,64,0
9,Validation,lymphomap,303,0


Review-only run; UPDATE_ADATA is False.
